In [2]:
# Defining project paths
BASE_DIR = "D:/Capstone/capstone_repo"
DATA_DIR = f"{BASE_DIR}/data"
NB_DIR = f"{BASE_DIR}/notebooks"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
import re

# Helper function: save as csv and gpkg files

In [88]:
def save_clean_data(df, gdf, filename):
    filepath = os.path.join(f"{DATA_DIR}/processed", filename)
    df.to_csv(filepath + ".csv", index=False)
    gdf.to_file(filepath + ".gpkg", layer="stops", driver="GPKG")

# Hospitals data

### Read data

In [89]:
hospitals = pd.read_csv(f"{DATA_DIR}/raw/casablanca_healthcare.csv")
hospitals.head()

,name,amenity,geometry,lat,lon
0,Clinique Badr مصحة بدر,clinic,POINT (-7.6410981 33.5948697),33.594870,-7.641098
1,clinique dentaire casablanca (cdc),clinic,POINT (-7.6482094 33.5874104),33.587410,-7.648209
2,Centre consultation et traitement dentaires,hospital,POINT (-7.6206013 33.5730773),33.573077,-7.620601
3,Clinique d'accouchement,clinic,POINT (-7.6240936 33.5930596),33.593060,-7.624094
4,Hôpital Sidi Othmane,hospital,POINT (-7.5733008 33.5585424),33.558542,-7.573301


### Missing names/values

In [90]:
# Manually filling in missing names based on the original dataset and Google Maps to ensure accuracy. 
hospitals.loc[47, "name"] = "Clinique Al Oumouma"
hospitals.loc[71, "name"] = "HOPITAL UNIVERSITAIRE DE PROXIMITE"
hospitals.loc[80, "name"] = "Hôpital Municipal sidi moumen Tacharouk"
hospitals.loc[88, "name"] = "Centre de Maladies du Rein et de Dialyse AL AMINE"
hospitals.loc[97, "name"] = "Clinique Al Wafaa"
hospitals.loc[98, "name"] = "Centre de Santé Al Massira 2 (Niveau 2)"

### Filtering Non-Hospital Healthcare Facilities (Dental, Ophtalmo, etc.)

Dropping any remaining rows with missing names, as they cannot be reliably identified.

In [91]:
# List of keywords to exclude
keywords = [
    "dentaire", "accouchement", "maternité", "pharmacie",
    "labo", "kine", "kiné", "ophtal",
    "hemo", "cardio", "radio",
    "Al Oumouma", "Dialyse", "Pasteur"
]

# Create regex pattern
pattern = "|".join(keywords)

# Drop null names
hospitals = hospitals.dropna(subset=["name"])

# Filter rows
hospitals = hospitals[~hospitals["name"].str.contains(pattern, case=False, na=False)]

### Duplicated rows

In [92]:
hospitals[hospitals.duplicated(subset=["name"], keep=False)]

,name,amenity,geometry,lat,lon
7,Avicenne Clinique des Spécialistes ابن سينا,clinic,POINT (-7.6110661 33.5542874),33.554287,-7.611066
8,Avicenne Clinique des Spécialistes ابن سينا,clinic,POINT (-7.6109113 33.5543506),33.554351,-7.610911
54,Hopital 20 Aout,hospital,"POLYGON ((-7.6207405 33.5752817, -7.6206681 33...",33.575137,-7.621018
55,Hopital 20 Aout,hospital,"POLYGON ((-7.6217004 33.57581, -7.6183329 33.5...",33.574653,-7.619653


Manually checked the accuracy of the location of each duplicated hospital. Kept the one with the most accurate location

In [93]:
hospitals = hospitals.drop_duplicates(subset=["name"], keep="last")

### Handling Arabic chars

Rows where name is only in arabic

In [94]:
# pattern: only Arabic letters + spaces
pattern = r'^[\u0600-\u06FF\s]+$'

arabic_only_df = hospitals[hospitals["name"].str.match(pattern, na=False)]

In [95]:
arabic_only_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   name      0 non-null      object 
 1   amenity   0 non-null      object 
 2   geometry  0 non-null      object 
 3   lat       0 non-null      float64
 4   lon       0 non-null      float64
dtypes: float64(2), object(3)
memory usage: 0.0+ bytes


No arabic only names: Good !

Removing arabic parts in some rows

In [96]:
def clean_name(text):
    text = str(text).strip()
    
    # Remove Arabic characters
    no_arabic = re.sub(r'[\u0600-\u06FF]+', '', text)
    no_arabic = re.sub(r'\s+', ' ', no_arabic).strip()
    
    # If something remains (French/Latin part exists)
    if no_arabic != "":
        return no_arabic
    else:
        # If it was fully Arabic, keep original
        return text

hospitals["name"] = hospitals["name"].apply(clean_name)

Renaming columns

In [97]:
hospitals = hospitals.rename(columns={
    "lat": "latitude",
    "lon": "longitude"
})

In [98]:
hospitals.drop(columns=["amenity", "geometry"], inplace=True)

In [99]:
hospitals.info()

<class 'pandas.core.frame.DataFrame'>
Index: 82 entries, 0 to 100
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       82 non-null     object 
 1   latitude   82 non-null     float64
 2   longitude  82 non-null     float64
dtypes: float64(2), object(1)
memory usage: 2.6+ KB


### Final processing by calling wrangle and save to files

In [100]:
hospitals["geometry"] = hospitals.apply(
    lambda row: Point(row["longitude"], row["latitude"]),
    axis=1
)

healthcare_gdf = gpd.GeoDataFrame(
    hospitals,
    geometry="geometry",
    crs="EPSG:4326"
)

In [101]:
save_clean_data(hospitals, healthcare_gdf, "Casablanca_Healthcare")

# Public Transport: Collected via JSON API

## CasaBus data

In [46]:
casabus = pd.read_csv(f"{DATA_DIR}/raw/CasaBus/CasaBus_stops.csv")
casabus.head()

,StopId,StopName,DirectionId,RouteId,LigneName,RouteColor,RouteTextColor,StopLat,StopLon
0,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,13,L013,1FE6E6,000000,33.592059,-7.633955
1,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,50,L050,990033,FFFFFF,33.592059,-7.633955
2,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,84,L084,0033FF,FFFFFF,33.592059,-7.633955
3,0:112 Boulevard D'Anfa,112 Boulevard D'Anfa,1,9E,L09E,0033FF,FFFFFF,33.592059,-7.633955
4,0:112 Boulevard D'Anfa_,112 Boulevard D'Anfa_,0,13,L013,1FE6E6,000000,33.592137,-7.633524


In [47]:
casabus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3723 entries, 0 to 3722
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   StopId          3723 non-null   object 
 1   StopName        3723 non-null   object 
 2   DirectionId     3723 non-null   int64  
 3   RouteId         3723 non-null   object 
 4   LigneName       3723 non-null   object 
 5   RouteColor      3723 non-null   object 
 6   RouteTextColor  3723 non-null   object 
 7   StopLat         3723 non-null   float64
 8   StopLon         3723 non-null   float64
dtypes: float64(2), int64(1), object(6)
memory usage: 261.9+ KB


Rename Columns

In [48]:
casabus = casabus.rename(columns={
    "StopId": "stop_id",
    "StopName": "stop_name",
    "StopLat": "latitude",
    "StopLon": "longitude",
    "LigneName": "Lines"
})

Remove Direction Duplicates

In [49]:
casabus["stop_name"] = casabus["stop_name"].str.replace("_", "", regex=False)

Aggregate Physical stops: Same stop appears multiple times because multiple routes serve it

In [51]:
casabus_grouped = (
    casabus
    .groupby(["stop_name", "latitude", "longitude"])
    .agg({
        "Lines": lambda x: list(x.unique())
    })
    .reset_index()
)

In [52]:
casabus_grouped["mode"] = "casabus"

In [56]:
casabus_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1778 entries, 0 to 1777
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   stop_name  1778 non-null   object 
 1   latitude   1778 non-null   float64
 2   longitude  1778 non-null   float64
 3   Lines      1778 non-null   object 
 4   mode       1778 non-null   object 
dtypes: float64(2), object(3)
memory usage: 69.6+ KB


## Casaway

In [62]:
lines = ["BW1", "BW2", "T1", "T2", "T3", "T4"]
stops_list = []
for line in lines:
    stops = pd.read_csv(f"{DATA_DIR}/raw/Casaway/Casaway_{line}_stops.csv")
    stops["Lines"] = line  # Add line information
    stops_list.append(stops)

In [63]:
casaway = pd.concat(stops_list, ignore_index=True)

In [64]:
casaway.head()

,StopPointRef,StopCity,StopStreet,StopStreetNumber,HandAccess,Lines,StopName.value,Location.Longitude,Location.Latitude
0,103,Casablanca,Boulevard El Qods,514.0,1,BW1,Al Inara,-7.59923,33.539640
1,110,Casablanca,Boulevard Mekdad Lahrizi,587.0,1,BW1,Al Joulane,-7.56217,33.543960
2,115,Casablanca,Route de Nouasseur,9.0,1,BW1,Al Moustakbal,-7.64672,33.531590
3,117,Casablanca,Boulevard Mohammed VI,1607.0,1,BW1,Al Qods,-7.58190,33.543840
4,104,Casablanca,Boulevard Mohammed VI,NaN,1,BW1,Amgala,-7.57905,33.539913


In [65]:
casaway.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   StopPointRef        157 non-null    int64  
 1   StopCity            157 non-null    object 
 2   StopStreet          157 non-null    object 
 3   StopStreetNumber    83 non-null     object 
 4   HandAccess          157 non-null    int64  
 5   Lines               157 non-null    object 
 6   StopName.value      157 non-null    object 
 7   Location.Longitude  157 non-null    float64
 8   Location.Latitude   157 non-null    float64
dtypes: float64(2), int64(2), object(5)
memory usage: 11.2+ KB


Renaming columns

In [66]:
casaway = casaway.rename(columns={
    "StopPointRef": "stop_id",
    "StopName.value": "stop_name",
    "Location.Latitude": "latitude",
    "Location.Longitude": "longitude",
    "HandAccess": "wheelchair_access"
})

Stations serving multiple lines

In [67]:
interchange_counts = (
    casaway.groupby("stop_id")["Lines"]
    .nunique()
    .reset_index(name="n_lines")
)

casaway = casaway.merge(interchange_counts, on="stop_id")

In [68]:
interchange_counts[interchange_counts["n_lines"] > 1]

,stop_id,n_lines
35,61,2


There is only one stop that serves two lines

In [69]:
casaway.iloc[casaway[casaway["n_lines"] > 1].index, [casaway.columns.get_loc("stop_name"), casaway.columns.get_loc("Lines")]]

,stop_name,Lines
56,Croisement Abd El Moumen,T1
97,Croisement Abd El Moumen,T2


In [71]:
casaway_grouped = (
    casaway
    .groupby(["stop_name", "latitude", "longitude"])
    .agg({
        "Lines": lambda x: list(x.unique())
    })
    .reset_index()
)

In [ ]:
def detect_mode(lines):
    """
    lines: list of strings, e.g. ['T1', 'T2'] or ['BW2']
    returns 'tram' if all lines start with 'T', 'busway' if all lines start with B/W, 'mixed' otherwise
    """
    if all(l.startswith("T") for l in lines):
        return "tram"
    elif all(l.startswith("BW") for l in lines): 
        return "busway"
    else:
        return "mixed"  # stop served by tram + busway

# Apply
casaway_grouped["mode"] = casaway_grouped["Lines"].apply(detect_mode)

In [76]:
casaway_grouped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   stop_name  152 non-null    object 
 1   latitude   152 non-null    float64
 2   longitude  152 non-null    float64
 3   Lines      152 non-null    object 
 4   mode       152 non-null    object 
dtypes: float64(2), object(3)
memory usage: 6.1+ KB


## Combining both dfs

In [116]:
stops_df = pd.concat([casabus_grouped, casaway_grouped], ignore_index=True)
stops_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1930 entries, 0 to 1929
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   stop_name  1930 non-null   object 
 1   latitude   1930 non-null   float64
 2   longitude  1930 non-null   float64
 3   Lines      1930 non-null   object 
 4   mode       1930 non-null   object 
dtypes: float64(2), object(3)
memory usage: 75.5+ KB


In [117]:
stops_df.head()

,stop_name,latitude,longitude,Lines,mode
0,112 Boulevard D'Anfa,33.592059,-7.633955,"[L013, L050, L084, L09E]",casabus
1,112 Boulevard D'Anfa,33.592137,-7.633524,"[L013, L050, L084, L09E]",casabus
2,4EME ARRONDISSEMENT,33.557130,-7.572611,"[L604, L067, L072]",casabus
3,4EME ARRONDISSEMENT,33.557659,-7.573472,"[L604, L067, L072]",casabus
4,ABATOIRS MUNICIPAUX,33.549595,-7.549733,"[L055, L068]",casabus


Converting to Geodataframe

In [118]:
stops_df["geometry"] = stops_df.apply(
    lambda row: Point(row["longitude"], row["latitude"]),
    axis=1
)

gdf_stops = gpd.GeoDataFrame(
    stops_df,
    geometry="geometry",
    crs="EPSG:4326"
)

Some stops are duplicated : opposite direction -> Spatial deduplication (merge stops within 20–30m)

WGS84 / UTM Zone 29N → EPSG:32629

In [103]:
gdf_stops_proj = gdf_stops.to_crs(epsg=32629)

Build 30m Spatial Clusters using spatial join

In [104]:
gdf_stops_proj["buffer"] = gdf_stops_proj.geometry.buffer(30)

In [105]:
joined = gpd.sjoin(
    gdf_stops_proj.set_geometry("buffer"),
    gdf_stops_proj,
    how="left",
    predicate="intersects"
)

Build clusters

In [108]:
G = nx.Graph()

for left_idx, row in joined.iterrows():
    right_idx = row["index_right"]
    
    if pd.notna(right_idx):
        G.add_edge(left_idx, right_idx)

clusters = list(nx.connected_components(G))

In [111]:
merged_stops = []

for cluster_id, cluster in enumerate(clusters):
    cluster_points = gdf_stops_proj.loc[list(cluster)]
    
    merged_stops.append({
        "cluster_id": cluster_id,
        "stop_name": cluster_points["stop_name"].iloc[0],
        "Lines": list(set(sum(cluster_points["Lines"], []))),
        "mode": list(cluster_points["mode"].unique()),
        "geometry": cluster_points.geometry.unary_union.centroid
    })

gdf_merged = gpd.GeoDataFrame(merged_stops, crs="EPSG:32629")

C:\Users\afafb\AppData\Local\Temp\ipykernel_28924\4051629887.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  "geometry": cluster_points.geometry.unary_union.centroid
C:\Users\afafb\AppData\Local\Temp\ipykernel_28924\4051629887.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  "geometry": cluster_points.geometry.unary_union.centroid
C:\Users\afafb\AppData\Local\Temp\ipykernel_28924\4051629887.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  "geometry": cluster_points.geometry.unary_union.centroid
C:\Users\afafb\AppData\Local\Temp\ipykernel_28924\4051629887.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  "geometry": cluster_points.geometry.unary_union.centroid
C:\Users\afafb\AppData\Local\Temp\ipykernel_28924\4051629887.py:11: DeprecationWarning: 

In [121]:
gdf_merged = gdf_merged.to_crs(epsg=4326)

In [122]:
gdf_merged.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1463 entries, 0 to 1462
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   cluster_id  1463 non-null   int64   
 1   stop_name   1463 non-null   object  
 2   Lines       1463 non-null   object  
 3   mode        1463 non-null   object  
 4   geometry    1463 non-null   geometry
dtypes: geometry(1), int64(1), object(3)
memory usage: 57.3+ KB


In [123]:
gdf_merged["longitude"] = gdf_merged.geometry.x
gdf_merged["latitude"] = gdf_merged.geometry.y

In [124]:
df_final = gdf_merged.drop(columns="geometry")

In [125]:
save_clean_data(df_final, gdf_merged, "Casablanca_Transport_Stops")